### Import Dependencies

In [1]:
from google.adk.agents import Agent 
from google.adk.models.lite_llm import LiteLlm  
import os 

from google.adk.sessions import InMemorySessionService 
from google.adk.runners import Runner 
from google.adk.agents.run_config import RunConfig  
from google.genai import types 

from utils.tools import check_warehouse_availability, reserve_warehouse_items 

/home/k/AI-Engineering-Bootcamp/01-ai-engineering-bootcamp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### ADK Agent

In [2]:
model = LiteLlm( 
    model = "openai/gpt-4.1-mini", 
    temperature = 0.0,  
    api_key = os.getenv("OPENAI_API_KEY"), 
) 

root_agent = Agent(  
    name = "warehouse_manager_agent", 
    model = model,  
    tools = [check_warehouse_availability, reserve_warehouse_items], 
    description="A agent that can check the availability of items in the warehouses and reserver them.", 
    instruction="""You are a part of the shopping assistant that can manage available inventory in the warehouses.

    You will be given a conversation history and a list of tools, your task is to perform actions requested by the latest user query. Answer part of the query that you can answer with the available tools.

    Instructions:
    - Use names specificly provided in the available tools. Don't add any additional text to the names.
    - You can run multipple tools at once.
    - Once you get the tool results back, you might choose to performa additional tool calls.
    - Once your suggested tool calls are done, set final_answer to True.
    - Never set final_answer to True if you are suggesting tool_calls.
    - As the final answer you should return an answer to the users query in a form of actions performed.
    - You must always check the availability of the items in the warehouses before reserving them.
    - Only reserve items in warehouses if entire order can be reserved or the user has confirmed that they want a partial reservation.
    - If you cannot reserve any items, return an answer that the order cannot be reserved.
    - If you can reserve some items, return an answer that the order can be partially reserved and include the details.
    - If only partial quantity can be reserved in some warehouses, try to combinethe required quantity from different warehouses.
    - Try to reserve items from the closest warehouse to the user first if users location is provided.
    """
)

### ADK Session

In [3]:
session_service = InMemorySessionService() 
await session_service.create_session( 
    app_name="warehouse_app", 
    user_id="user_1", 
    session_id="session_2", 
)

runner = Runner( 
    agent=root_agent, 
    app_name="warehouse_app", 
    session_service=session_service, 
) 

In [4]:
message = types.Content( 
    role="user", 
    parts=[types.Part(text="What is the availability of B0C4DB5WGW in all of the warehouses?")],
)

In [5]:
result = runner.run( 
    user_id="user_1", 
    session_id="session_2", 
    new_message=message, 
    run_config=RunConfig( 
        max_llm_calls=1,
    ),
)

In [6]:
for event in result: 
    print(event) 

model_version='gpt-4.1-mini-2025-04-14' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'items': [
            {<... 2 items at Max depth ...>},
          ]
        },
        id='call_tRVhRQ6IxeG1hgunj3w6lqOR',
        name='check_warehouse_availability'
      )
    ),
  ],
  role='model'
) grounding_metadata=None partial=False turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  cached_content_token_count=0,
  candidates_token_count=33,
  prompt_token_count=624,
  total_token_count=657
) live_session_resumption_update=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None invocation_id='e-c9d2d6c8-1b97-4829-be7c-82f0a5714bce' author='warehouse_manager_agent' actions=EventActions(skip_summarization=None, state_delta={}, ar

Exception in thread Thread-30 (_asyncio_thread_main):
Traceback (most recent call last):
  File "/home/k/.pyenv/versions/3.12.11/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/home/k/AI-Engineering-Bootcamp/01-ai-engineering-bootcamp/.venv/lib/python3.12/site-packages/ipykernel/ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "/home/k/.pyenv/versions/3.12.11/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/home/k/AI-Engineering-Bootcamp/01-ai-engineering-bootcamp/.venv/lib/python3.12/site-packages/google/adk/runners.py", line 347, in _asyncio_thread_main
    asyncio.run(_invoke_run_async())
  File "/home/k/.pyenv/versions/3.12.11/lib/python3.12/asyncio/runners.py", line 195, in run
    return runner.run(main)
           ^^^^^^^^^^^^^^^^
  File "/home/k/.pyenv/versions/3.12.11/lib/python3.12/asyncio/runners.py", line 118, in run
    return self._loop.run_until_complet

model_version=None content=Content(
  parts=[
    Part(
      function_response=FunctionResponse(
        id='call_tRVhRQ6IxeG1hgunj3w6lqOR',
        name='check_warehouse_availability',
        response={
          'can_fulfill_completely': False,
          'details': [
            {<... 6 items at Max depth ...>},
            {<... 6 items at Max depth ...>},
            {<... 6 items at Max depth ...>},
            {<... 6 items at Max depth ...>},
            {<... 6 items at Max depth ...>},
            <... 1 more items ...>,
          ],
          'unavailable_items': [
            {<... 4 items at Max depth ...>},
          ],
          'warehouses_full_fulfillment': [],
          'warehouses_partial_fulfillment': []
        }
      )
    ),
  ],
  role='user'
) grounding_metadata=None partial=None turn_complete=None finish_reason=None error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=None live_session_resumption_update=None input_transcrip

In [10]:
session_service = InMemorySessionService() 
await session_service.create_session( 
    app_name="warehouse_app", 
    user_id="user_1", 
    session_id="session_11", 
)

runner = Runner( 
    agent=root_agent, 
    app_name="warehouse_app", 
    session_service=session_service, 
) 

message = types.Content( 
    role="user", 
    parts=[types.Part(text="What is the availability of B0C4DBSWGW in all of the warehouses?")],
)

result = runner.run( 
    user_id="user_1", 
    session_id="session_11", 
    new_message=message, 
    run_config=RunConfig( 
        max_llm_calls=3,
    ),
)

In [11]:
for event in result: 
    print(event) 

model_version='gpt-4.1-mini-2025-04-14' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'items': [
            {<... 2 items at Max depth ...>},
          ]
        },
        id='call_nDkt56z7UGhbapNYhwkWWZCF',
        name='check_warehouse_availability'
      )
    ),
  ],
  role='model'
) grounding_metadata=None partial=False turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  cached_content_token_count=0,
  candidates_token_count=32,
  prompt_token_count=623,
  total_token_count=655
) live_session_resumption_update=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None invocation_id='e-171d1241-1a49-4139-9c4d-60e88036854c' author='warehouse_manager_agent' actions=EventActions(skip_summarization=None, state_delta={}, ar

Task exception was never retrieved
future: <Task finished name='Task-52' coro=<LoggingWorker._worker_loop() done, defined at /home/k/AI-Engineering-Bootcamp/01-ai-engineering-bootcamp/.venv/lib/python3.12/site-packages/litellm/litellm_core_utils/logging_worker.py:61> exception=RuntimeError('<Queue at 0x7c832d39d730 maxsize=50000> is bound to a different event loop')>
Traceback (most recent call last):
  File "/home/k/AI-Engineering-Bootcamp/01-ai-engineering-bootcamp/.venv/lib/python3.12/site-packages/litellm/litellm_core_utils/logging_worker.py", line 69, in _worker_loop
    task = await self._queue.get()
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/k/.pyenv/versions/3.12.11/lib/python3.12/asyncio/queues.py", line 155, in get
    getter = self._get_loop().create_future()
             ^^^^^^^^^^^^^^^^
  File "/home/k/.pyenv/versions/3.12.11/lib/python3.12/asyncio/mixins.py", line 20, in _get_loop
    raise RuntimeError(f'{self!r} is bound to a different event loop')
RuntimeError: <

model_version='gpt-4.1-mini-2025-04-14' content=Content(
  parts=[
    Part(
      text="""The product B0C4DBSWGW is available in the following warehouses with full fulfillment capability:
- Lyon Regional Warehouse (Lyon, France) with 54 units available
- Munich Logistics Hub (Munich, Germany) with 43 units available
- Paris Central Depot (Paris, France) with 71 units available
- Hamburg North Warehouse (Hamburg, Germany) with 76 units available

It is not available in Berlin Distribution Center and Marseille Mediterranean Hub. Let me know if you want to reserve any quantity from these warehouses.
final_answer: True"""
    ),
  ],
  role='model'
) grounding_metadata=None partial=False turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  cached_content_token_count=0,
  candidates_token_count=116,
  prompt_token_count=1386,
  total_token_count=1502
) liv

In [12]:
session_service = InMemorySessionService() 
await session_service.create_session( 
    app_name="warehouse_app", 
    user_id="user_1", 
    session_id="session_12", 
)

runner = Runner( 
    agent=root_agent, 
    app_name="warehouse_app", 
    session_service=session_service, 
) 

message = types.Content( 
    role="user", 
    parts=[types.Part(text="What is the availability of B0C4DBSWGW in all of the warehouses?")],
)

result = runner.run( 
    user_id="user_1", 
    session_id="session_12", 
    new_message=message, 
    run_config=RunConfig( 
        max_llm_calls=3,
    ),
)

In [13]:
for event in result: 
    if event.is_final_response():
        print("Final Response:")
        print(event.content.parts[0].text) 

Task exception was never retrieved
future: <Task finished name='Task-69' coro=<LoggingWorker._worker_loop() done, defined at /home/k/AI-Engineering-Bootcamp/01-ai-engineering-bootcamp/.venv/lib/python3.12/site-packages/litellm/litellm_core_utils/logging_worker.py:61> exception=RuntimeError('<Queue at 0x7c832d39d730 maxsize=50000> is bound to a different event loop')>
Traceback (most recent call last):
  File "/home/k/AI-Engineering-Bootcamp/01-ai-engineering-bootcamp/.venv/lib/python3.12/site-packages/litellm/litellm_core_utils/logging_worker.py", line 69, in _worker_loop
    task = await self._queue.get()
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/k/.pyenv/versions/3.12.11/lib/python3.12/asyncio/queues.py", line 155, in get
    getter = self._get_loop().create_future()
             ^^^^^^^^^^^^^^^^
  File "/home/k/.pyenv/versions/3.12.11/lib/python3.12/asyncio/mixins.py", line 20, in _get_loop
    raise RuntimeError(f'{self!r} is bound to a different event loop')
RuntimeError: <

Final Response:
The product B0C4DBSWGW is available for complete fulfillment in the following warehouses:
- Lyon Regional Warehouse (Lyon, France) with 54 units available
- Munich Logistics Hub (Munich, Germany) with 43 units available
- Paris Central Depot (Paris, France) with 71 units available
- Hamburg North Warehouse (Hamburg, Germany) with 76 units available

It is not available in Berlin Distribution Center and Marseille Mediterranean Hub.


In [ ]:
query = "Can you reserve 5 items of B0CF57H28T in the French office?" 
message = types.Content( 
    role="user", 
    parts=[types.Part(text=query)],
)

result = runner.run( 
    user_id="user_1", 
    session_id="session_2", 
    new_message=message, 
    run_config=RunConfig( 
        max_llm_calls=3,
    ),
)

In [ ]:
result

In [ ]:
for data in result: 
    print(data)